In [ ]:
!git clone https://github.com/ultralytics/ultralytics


fatal: destination path 'ultralytics' already exists and is not an empty directory.


In [ ]:
# %cd content
%cd ultralytics
# %cd ..
# !pip install -e .

/content/ultralytics


In [ ]:
import os

def print_tree(root, prefix=""):
    # list all items sorted
    items = sorted(os.listdir(root))
    for i, item in enumerate(items):
        path = os.path.join(root, item)
        connector = "└── " if i == len(items)-1 else "├── "
        print(prefix + connector + item)
        if os.path.isdir(path):
            extension = "    " if i == len(items)-1 else "│   "
            print_tree(path, prefix + extension)

# change this to your folder path
models_path = "/content/ultralytics/ultralytics/cfg/models"
print_tree(models_path)

├── 11
│   ├── .ipynb_checkpoints
│   ├── yolo11-cls-resnet18.yaml
│   ├── yolo11-cls.yaml
│   ├── yolo11-lol.yaml
│   ├── yolo11-obb.yaml
│   ├── yolo11-pose.yaml
│   ├── yolo11-seg.yaml
│   ├── yolo11.yaml
│   ├── yoloe-11-seg.yaml
│   └── yoloe-11.yaml
├── 12
│   ├── yolo12-cls.yaml
│   ├── yolo12-obb.yaml
│   ├── yolo12-pose.yaml
│   ├── yolo12-seg.yaml
│   └── yolo12.yaml
├── 26
│   ├── yolo26-cls.yaml
│   ├── yolo26-obb.yaml
│   ├── yolo26-p2.yaml
│   ├── yolo26-p6.yaml
│   ├── yolo26-pose.yaml
│   ├── yolo26-seg.yaml
│   ├── yolo26.yaml
│   ├── yoloe-26-seg.yaml
│   └── yoloe-26.yaml
├── README.md
├── rt-detr
│   ├── rtdetr-l.yaml
│   ├── rtdetr-resnet101.yaml
│   ├── rtdetr-resnet50.yaml
│   └── rtdetr-x.yaml
├── v10
│   ├── yolov10b.yaml
│   ├── yolov10l.yaml
│   ├── yolov10m.yaml
│   ├── yolov10n.yaml
│   ├── yolov10s.yaml
│   └── yolov10x.yaml
├── v3
│   ├── yolov3-spp.yaml
│   ├── yolov3-tiny.yaml
│   └── yolov3.yaml
├── v5
│   ├── yolov5-p6.yaml
│   └── yolov5.yaml
├── v6


In [ ]:
# !pip install -q ultralytics

import ultralytics
ultralytics.checks()
print(ultralytics.__file__)

Ultralytics 8.4.30 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 21.3/107.7 GB disk)
/content/ultralytics/ultralytics/__init__.py


# Inject files

In [ ]:
with open("/content/ultralytics/ultralytics/nn/tasks.py", "r") as f:
    content = f.read()

# 1. Add import at the top
content = content.replace(
    "from ultralytics.nn.modules import (",
    "from ultralytics.nn.modules.block import C3k2_CBAM\nfrom ultralytics.nn.modules import ("
)

# 2. Add to base_modules (after C3k2)
content = content.replace(
    "            C3k2,\n            RepNCSPELAN4,",
    "            C3k2,\n            C3k2_CBAM,\n            RepNCSPELAN4,"
)

# 3. Add to repeat_modules (after C3k2)
content = content.replace(
    "            C3k2,\n            C2fAttn,",
    "            C3k2,\n            C3k2_CBAM,\n            C2fAttn,"
)

# 4. C3k2 special case for M/L/X scales
content = content.replace(
    "            if m is C3k2:  # for M/L/X sizes",
    "            if m in {C3k2, C3k2_CBAM}:  # for M/L/X sizes"
)

with open("/content/ultralytics/ultralytics/nn/tasks.py", "w") as f:
    f.write(content)

# Verify
import subprocess
result = subprocess.run(["grep", "-n", "C3k2_CBAM", "/content/ultralytics/ultralytics/nn/tasks.py"], capture_output=True, text=True)
print(result.stdout)

14:from ultralytics.nn.modules.block import C3k2_CBAM
15:from ultralytics.nn.modules.block import C3k2_CBAM
1596:            C3k2_CBAM,
1623:            C3k2_CBAM,
1661:            if m in {C3k2, C3k2_CBAM}:  # for M/L/X sizes



In [ ]:
file_path = "/content/ultralytics/ultralytics/nn/modules/block.py"

with open(file_path, "r") as f:
    content = f.read()

# Remove any broken previous injections
import re
content = re.sub(r"class C3k2_CBAM[\s\S]*?return .*?\n", "", content)

# Now inject fresh
cbam_code = """
# CBAM attention modules
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv1(x))


class CBAMBlock(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.ca = ChannelAttention(in_channels)
        self.sa = SpatialAttention()

    def forward(self, x):
        x = self.ca(x) * x
        x = self.sa(x) * x
        return x


class C3k2_CBAM(nn.Module):
    def __init__(self, c1, c2, n=1, c3k=False, e=0.5, attn=False, g=1, shortcut=True):
        super().__init__()
        self.c3k2 = C3k2(c1, c2, n=n, c3k=c3k, e=e, attn=attn, g=g, shortcut=shortcut)
        self.cbam = CBAMBlock(c2)

    def forward(self, x):
        return self.cbam(self.c3k2(x))
"""

content += cbam_code

with open(file_path, "w") as f:
    f.write(content)

print("Re-injected cleanly ✅")

Re-injected cleanly ✅


In [ ]:
file_path = "/content/ultralytics/ultralytics/nn/modules/block.py"

with open(file_path, "r") as f:
    lines = f.readlines()

if not any("C3k2_CBAM" in line for line in lines):
    for i, line in enumerate(lines):
        if "__all__" in line and "(" in line:
            lines.insert(i + 1, '    "C3k2_CBAM",\n')
            print(f"Inserted at line {i+1}")
            break

    with open(file_path, "w") as f:
        f.writelines(lines)

    print("Added C3k2_CBAM to __all__")
else:
    print("Already present")

Already present


In [ ]:
file_path = "/content/ultralytics/ultralytics/nn/modules/block.py"

# Step 1: Read
with open(file_path, "r") as f:
    content = f.read()

print("Before change contains C3k2_CBAM:", "C3k2_CBAM" in content)

# Step 2: FORCE change (no condition, no replace)
content = content.replace(
    '"C3k2",',
    '"C3k2",\n    "C3k2_CBAM",'
)

# Step 3: Write
with open(file_path, "w") as f:
    f.write(content)

# Step 4: Read again immediately
with open(file_path, "r") as f:
    new_content = f.read()

print("After change contains C3k2_CBAM:", "C3k2_CBAM" in new_content)

Before change contains C3k2_CBAM: True
After change contains C3k2_CBAM: True


In [ ]:
!grep -n "class C3k2_CBAM" /content/ultralytics/ultralytics/nn/modules/block.py

2160:class C3k2_CBAM(nn.Module):


In [ ]:
# %cd ultralytics
!pip install -e .

/content/ultralytics
Obtaining file:///content/ultralytics
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ultralytics (pyproject.toml) ... done
  Created wheel for ultralytics: filename=ultralytics-8.4.30-0.editable-py3-none-any.whl size=23097 sha256=f5d67ac09ef15c344c0435a0e91f64fd1646f832a3f67a04997318a5a679b73f
  Stored in directory: /tmp/pip-ephem-wheel-cache-04486p2p/wheels/60/e0/59/e2f034f296abbdca5c21e3f5be76b9ca685f13c7bd17f8b58c
Successfully built ultralytics
  Attempting uninstall: ultralytics
    Found existing installation: ultralytics 8.4.30
    Uninstalling ultralytics-8.4.30:
      Successfully uninstalled ultralytics-8.4.30


In [ ]:
import ultralytics.nn.modules.block as block
print(hasattr(block, "C3k2_CBAM"))
print(block.__file__)

True
/content/ultralytics/ultralytics/nn/modules/block.py


In [ ]:
yaml_content = """# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license

# YOLO11 object detection model with CBAM attention
# Parameters
nc: 80 # number of classes
scales:
  n: [0.50, 0.25, 1024]
  s: [0.50, 0.50, 1024]
  m: [0.50, 1.00, 512]
  l: [1.00, 1.00, 512]
  x: [1.00, 1.50, 512]

# YOLO11 backbone with CBAM
backbone:
  - [-1, 1, Conv, [64, 3, 2]]           # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]]          # 1-P2/4
  - [-1, 2, C3k2_CBAM, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]]          # 3-P3/8
  - [-1, 2, C3k2_CBAM, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]]          # 5-P4/16
  - [-1, 2, C3k2_CBAM, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]         # 7-P5/32
  - [-1, 2, C3k2_CBAM, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]            # 9
  - [-1, 2, C2PSA, [1024]]              # 10

# YOLO11 head with CBAM
head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]           # cat backbone P4
  - [-1, 2, C3k2_CBAM, [512, False]]    # 13

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]           # cat backbone P3
  - [-1, 2, C3k2_CBAM, [256, False]]    # 16 (P3/8-small)

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]          # cat head P4
  - [-1, 2, C3k2_CBAM, [512, False]]    # 19 (P4/16-medium)

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]          # cat head P5
  - [-1, 2, C3k2_CBAM, [1024, True]]   # 22 (P5/32-large)

  - [[16, 19, 22], 1, Detect, [nc]]     # Detect(P3, P4, P5)
"""

import os
os.makedirs("/content/ultralytics/ultralytics/cfg/models/11", exist_ok=True)

with open("/content/ultralytics/ultralytics/cfg/models/11/yolo11-lol.yaml", "w") as f:
    f.write(yaml_content)

print("Created /content/ultralytics/ultralytics/cfg/models/11/yolo11-lol.yaml")

# Verify
with open("/content/ultralytics/ultralytics/cfg/models/11/yolo11-lol.yaml", "r") as f:
    print(f.read())

Created /content/ultralytics/ultralytics/cfg/models/11/yolo11-lol.yaml
# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license

# YOLO11 object detection model with CBAM attention
# Parameters
nc: 80 # number of classes
scales:
  n: [0.50, 0.25, 1024]
  s: [0.50, 0.50, 1024]
  m: [0.50, 1.00, 512]
  l: [1.00, 1.00, 512]
  x: [1.00, 1.50, 512]

# YOLO11 backbone with CBAM
backbone:
  - [-1, 1, Conv, [64, 3, 2]]           # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]]          # 1-P2/4
  - [-1, 2, C3k2_CBAM, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]]          # 3-P3/8
  - [-1, 2, C3k2_CBAM, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]]          # 5-P4/16
  - [-1, 2, C3k2_CBAM, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]         # 7-P5/32
  - [-1, 2, C3k2_CBAM, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]            # 9
  - [-1, 2, C2PSA, [1024]]              # 10

# YOLO11 head with CBAM
head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]   

In [ ]:
from ultralytics.nn.modules import *
from ultralytics.nn.modules.block import C3k2_CBAM

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install torchinfo

In [ ]:
import os
import ultralytics
import yaml
from ultralytics import YOLO
from torchinfo import summary

# Copier les données en local (accélère entrainement)
dossier_drive = "/content/drive/MyDrive/yolo-lol/Projet_IRM/Donnees_RoboFlow"
dossier_local = "/content/Donnees_RoboFlow"

In [ ]:
if not os.path.exists(dossier_local):
    print("Copie des données en cours vers le disque local")
    !cp -r "{dossier_drive}" "{dossier_local}"
    print("Copie terminée !")
else:
    print("Les données sont déjà sur le disque local.")

#Configurer chemins locaux
dataset_path = '/content/Donnees_RoboFlow'
yaml_path = os.path.join(dataset_path, 'data.yaml')

# Lire et mettre à jour le fichier YAML
with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

# On s'assure que YOLO pointe vers le disque local
data['train'] = os.path.join(dataset_path, 'train/images')
data['val'] = os.path.join(dataset_path, 'valid/images')
data['test'] = os.path.join(dataset_path, 'test/images')

with open(yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

# Charger le modèle YOLO11 Small
# model_orig = YOLO('yolo11s.pt')
model_orig = YOLO('/content/ultralytics/ultralytics/cfg/models/11/yolo11.yaml')
model_orig.load('yolo11s.pt')

# Path to your modified YAML
model_yaml = '/content/ultralytics/ultralytics/cfg/models/11/yolo11-lol.yaml'

import sys
import importlib

# First, remove all ultralytics related modules from sys.modules
# This is a more aggressive approach to ensure all references are cleared.
modules_to_delete = [
    name for name in sys.modules if name.startswith('ultralytics')
]
print(f"Clearing {len(modules_to_delete)} ultralytics modules from sys.modules...")
for name in modules_to_delete:
    del sys.modules[name]

# Now, re-import necessary components
import ultralytics
import ultralytics.nn.tasks as tasks
from ultralytics.nn.modules.block import C3k2_CBAM
from ultralytics import YOLO # Re-import YOLO after clearing all ultralytics modules

tasks.__dict__['C3k2_CBAM'] = C3k2_CBAM

print('C3k2_CBAM' in globals())


model = YOLO(model_yaml)                           # now YOLO can see it

# Initialize the model from scratch (random weights)
model_custom = YOLO(model_yaml)
model_custom.load('yolo11s.pt')

Les données sont déjà sur le disque local.
WARNING ⚠️ no model scale passed. Assuming scale='n'.
WARNING ⚠️ WARNING ⚠️ no model scale passed. Assuming scale='n'.
Transferred 119/499 items from pretrained weights
Transferred 119/499 items from pretrained weights
Clearing 118 ultralytics modules from sys.modules...
True
WARNING ⚠️ no model scale passed. Assuming scale='n'.
WARNING ⚠️ WARNING ⚠️ no model scale passed. Assuming scale='n'.
WARNING ⚠️ WARNING ⚠️ WARNING ⚠️ no model scale passed. Assuming scale='n'.
WARNING ⚠️ no model scale passed. Assuming scale='n'.
WARNING ⚠️ WARNING ⚠️ no model scale passed. Assuming scale='n'.
WARNING ⚠️ WARNING ⚠️ WARNING ⚠️ no model scale passed. Assuming scale='n'.
Transferred 72/523 items from pretrained weights
Transferred 72/523 items from pretrained weights
Transferred 72/523 items from pretrained weights


YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2_CBAM(
        (c3k2): C3k2(
          (cv1): Conv(
            (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
            (act): SiLU(inplace=True)
          )
          (cv2): Conv(
            (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (bn): BatchNorm2d(64, eps=0.001,

In [ ]:
summary(model_orig.model, input_size=(1, 3, 640, 640), col_names=["input_size", "output_size", "num_params"])

Layer (type:depth-idx)                                       Input Shape               Output Shape              Param #
DetectionModel                                               [1, 3, 640, 640]          [1, 84, 8400]             --
├─Sequential: 1-1                                            --                        --                        --
│    └─Conv: 2-1                                             [1, 3, 640, 640]          [1, 16, 320, 320]         --
│    │    └─Conv2d: 3-1                                      [1, 3, 640, 640]          [1, 16, 320, 320]         432
│    │    └─BatchNorm2d: 3-2                                 [1, 16, 320, 320]         [1, 16, 320, 320]         32
│    └─Detect: 2-122                                         --                        --                        (recursive)
│    │    └─ModuleList: 3-142                                --                        --                        (recursive)
│    └─Conv: 2-3                                

In [ ]:
summary(model_custom.model, input_size=(1, 3, 640, 640), col_names=["input_size", "output_size", "num_params"])

Layer (type:depth-idx)                                            Input Shape               Output Shape              Param #
DetectionModel                                                    [1, 3, 640, 640]          [1, 84, 8400]             --
├─Sequential: 1-1                                                 --                        --                        --
│    └─Conv: 2-1                                                  [1, 3, 640, 640]          [1, 16, 320, 320]         --
│    │    └─Conv2d: 3-1                                           [1, 3, 640, 640]          [1, 16, 320, 320]         432
│    │    └─BatchNorm2d: 3-2                                      [1, 16, 320, 320]         [1, 16, 320, 320]         32
│    └─Detect: 2-129                                              --                        --                        (recursive)
│    │    └─ModuleList: 3-150                                     --                        --                        (recursive)
│    └─C

In [ ]:
#POUR ENTRAINER LE MODELE
# Lancer l'entraînement
print("\nDébut de l'entraînement")
results = model_custom.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    project='/content/drive/MyDrive/yolo-lol/Projet_IRM',
    name='yolo11_lol',
    seed=42,
    close_mosaic=10,
    optimizer='AdamW',
    lr0=0.001
)

print("Entraînement terminé.")


Début de l'entraînement
Ultralytics 8.4.30 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Ultralytics 8.4.30 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Ultralytics 8.4.30 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Donnees_RoboFlow/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/50      2.43G      2.941      4.392      2.657         24        640: 100% ━━━━━━━━━━━━ 153/153 2.2it/s 1:09
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 34/34 2.5it/s 13.5s
                   all       1074       1224          0          0          0          0
                   all       1074       1224          0          0          0          0
                   all       1074       1224          0          0          0          0

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/50      2.95G      2.293      3.853      2.243         32        640: 0% ──────────── 0/153  0.2s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/50      2.95G      1.984      3.007      1.986         21        640: 100% ━━━━━━━━━━━━ 153/153 3.2it/s 47.8s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 34/34 3.7it/s 9.2s
                   all       1074       1224      0.503      0.256      0.296      0.141
                   all       1074       1224      0.503      0.256      0.296      0.141
                   all       1074       1224      0.503      0.256      0.296      0.141

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/50      2.95G      1.878       2.96      1.892         25        640: 0% ──────────── 0/153  0.2s

...

      48/50      2.95G     0.7897      0.567      1.087         13        640: 100% ━━━━━━━━━━━━ 153/153 3.4it/s 45.6s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 34/34 4.0it/s 8.6s
                   all       1074       1224      0.926      0.877      0.947      0.738
                   all       1074       1224      0.926      0.877      0.947      0.738
                   all       1074       1224      0.926      0.877      0.947      0.738

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      49/50      2.95G     0.8345     0.5631      1.192         16        640: 0% ──────────── 0/153  0.2s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      49/50      2.95G     0.7753     0.5597      1.081         12        640: 100% ━━━━━━━━━━━━ 153/153 3.4it/s 45.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 34/34 4.1it/s 8.3s
                   all       1074       1224      0.909      0.894      0.947      0.746
                   all       1074       1224      0.909      0.894      0.947      0.746
                   all       1074       1224      0.909      0.894      0.947      0.746

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      50/50      2.95G     0.6058     0.5583     0.9057         19        640: 0% ──────────── 0/153  0.2s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      50/50      2.95G     0.7774     0.5535      1.077         13        640: 100% ━━━━━━━━━━━━ 153/153 3.3it/s 45.7s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 34/34 3.7it/s 9.2s
                   all       1074       1224      0.921      0.891      0.948      0.745
                   all       1074       1224      0.921      0.891      0.948      0.745
                   all       1074       1224      0.921      0.891      0.948      0.745

50 epochs completed in 0.800 hours.

50 epochs completed in 0.800 hours.

50 epochs completed in 0.800 hours.
Optimizer stripped from /content/drive/MyDrive/yolo-lol/Projet_IRM/yolo11_lol2/weights/last.pt, 5.6MB
Optimizer stripped from /content/drive/MyDrive/yolo-lol/Projet_IRM/yolo11_lol2/weights/last.pt, 5.6MB
Optimizer stripped from /content/drive/MyDrive/yolo-lol/Projet_IRM/yolo11_lol2/weights/last.pt, 5.6MB
Optimizer stripped from /content/drive/MyDrive/yolo-lol/Projet_IRM/yol

In [ ]:
#POUR UTILISER LE MODELE AVEC LES MEILLEURS POIDS ENTRAINÉS

# 1. On reconnecte le Drive
from google.colab import drive
from ultralytics import YOLO

drive.mount('/content/drive')

weights_path = '/content/drive/MyDrive/Colab Notebooks/Projet_IRM/baseline_yolo11/weights/best.pt'
model = YOLO(weights_path)

print("Modèle chargé avec poids modifiés")

In [ ]:
#POUR ENTRAINER LE MODELE
# Lancer l'entraînement
print("\nDébut de l'entraînement")
results = model_orig.model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    project='/content/drive/MyDrive/Colab Notebooks/Projet_IRM',
    name='baseline_yolo11',
    seed=42,
    close_mosaic=10,
    optimizer='AdamW',
    lr0=0.001
)

print("Entraînement terminé.")

In [ ]:
#POUR UTILISER LE MODELE AVEC LES MEILLEURS POIDS ENTRAINÉS

# 1. On reconnecte le Drive
from google.colab import drive
from ultralytics import YOLO

drive.mount('/content/drive')

weights_path = '/content/drive/MyDrive/Colab Notebooks/Projet_IRM/baseline_yolo11/weights/best.pt'
model = YOLO(weights_path)

print("Modèle chargé avec poids modifiés")